# Developing an AI-Powered Financial Chatbot
**BCG GenAI Job Simulation (Forage), Task 2**
**Prepared by:** Gibson Nyendwa, Junior Data Scientist, GenAI Consulting Team
**For:** Aisha, Senior Data Scientist | **Client:** Global Finance Corp. (GFC)

---

## Objective
Turn the Task 1 analysis of Microsoft, Tesla and Apple (FY2023 to FY2025) into a chatbot that answers financial questions in plain English, meeting GFC's four requirements: **efficiency, accuracy, a user-friendly interface and scalability**.

## Approach
I built the chatbot in two versions:
1. **Version 1: predefined queries.** A minimal rule-based bot that matches exact questions with `if`/`elif` logic. It proves the concept and shows where the project started.
2. **Version 2: the GFC Financial Chatbot.** A rule-based natural language understanding pipeline that handles questions phrased in many ways, follow-up questions, comparisons and health summaries, and explains one-off items rather than reporting bare numbers.

The code lives in `chatbot/financial_chatbot.py`, with 32 automated tests in `tests/test_chatbot.py`.

## 1. Load the Task 1 dataset

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
from chatbot.financial_chatbot import FinancialChatbot, simple_chatbot, METRICS, CONTEXT_NOTES

df = pd.read_csv('../data/financial_analysis.csv')
df[['Company', 'Fiscal Year', 'Total Revenue', 'Net Income', 'Revenue Growth (%)', 'Net Income Growth (%)']]

,Company,Fiscal Year,Total Revenue,Net Income,Revenue Growth (%),Net Income Growth (%)
0,Apple,2023,383285,96995,NaN,NaN
1,Apple,2024,391035,93736,2.02,-3.36
2,Apple,2025,416161,112010,6.43,19.50
3,Microsoft,2023,211915,72361,NaN,NaN
4,Microsoft,2024,245122,88136,15.67,21.80
5,Microsoft,2025,281724,101832,14.93,15.54
6,Tesla,2023,96773,14997,NaN,NaN
7,Tesla,2024,97690,7091,0.95,-52.72
8,Tesla,2025,94827,3794,-2.93,-46.50


## 2. Integrating the financial data with the chatbot
Three techniques turn the static Task 1 data into answers the chatbot can retrieve on demand.

**Data structuring.** The data is stored in a tidy table, one row per company per fiscal year, in CSV (which the chatbot reads) and JSON (nested by company, then year, ready for a web or API front end). Either way, any figure can be found with just three keys: **company, year and metric**.

In [2]:
import json

# JSON version, nested company -> year -> metric, for a future web or API front end
nested = {
    company: {
        int(row['Fiscal Year']): {m['label']: float(row[m['column']]) for m in METRICS.values()}
        for _, row in group.iterrows()
    }
    for company, group in df.groupby('Company')
}
with open('../data/financial_data.json', 'w') as f:
    json.dump(nested, f, indent=2)

print(json.dumps({'Tesla': {2025: nested['Tesla'][2025]}}, indent=2))

{
  "Tesla": {
    "2025": {
      "total revenue": 94827.0,
      "net income": 3794.0,
      "total assets": 137806.0,
      "total liabilities": 54941.0,
      "operating cash flow": 14747.0,
      "net profit margin": 4.0,
      "liabilities-to-assets ratio": 39.87,
      "shareholders' equity": 82865.0
    }
  }
}


**Predefined data points.** Each kind of question is mapped to one specific column in the dataset. This mapping lives in the chatbot's `METRICS` dictionary: the words a user might type on the left, the exact data point retrieved on the right.

In [3]:
mapping = pd.DataFrame(
    [(', '.join(m['keywords'][:3]), m['column']) for m in METRICS.values()],
    columns=['If the user mentions...', '...the chatbot retrieves this column'])
mapping

,If the user mentions...,...the chatbot retrieves this column
0,"total revenue, revenue, revenues",Total Revenue
1,"net income, net profit, profit",Net Income
2,"total assets, assets",Total Assets
3,"total liabilities, liabilities, debt",Total Liabilities
4,"cash flow from operating activities, operating...",Cash Flow from Operating Activities
5,"net profit margin, profit margin, net margin",Net Margin (%)
6,"liabilities to assets, debt ratio, leverage",Liabilities to Assets (%)
7,"shareholders equity, shareholders' equity, equity",Shareholders Equity


**Retrieval method.** Once the company, year and metric are identified, retrieval is a simple filter on the table. This is the same lookup the chatbot uses internally.

In [4]:
def retrieve(company, year, column):
    row = df[(df['Company'] == company) & (df['Fiscal Year'] == year)]
    return row[column].iloc[0]

# User asks: "What was Tesla's net income in 2025?"
# -> company = Tesla, year = 2025, metric = net income -> column 'Net Income'
print("Tesla, FY2025, Net Income:", retrieve('Tesla', 2025, 'Net Income'), "(US$ millions)")
print("Chatbot reply:", FinancialChatbot().respond("What was Tesla's net income in 2025?"))

Tesla, FY2025, Net Income: 3794 (US$ millions)
Chatbot reply: Tesla's net income in FY2025 was $3.8bn, which fell 46.5% on FY2024.
  Context: Tesla linked the fall to lower vehicle deliveries and regulatory credit revenue, tariffs, and higher spending on AI and R&D.
  You could also ask: "How has Tesla's net income changed?" or "Which company has the highest net income?"


## 3. Version 1: predefined queries
The simplest chatbot: each supported question is matched exactly and mapped to an answer calculated from the data. Anything else gets a polite refusal.

In [5]:
predefined = [
    "What is the total revenue?",
    "How has net income changed over the last year?",
    "What are the total assets?",
    "What are the total liabilities?",
    "What is the cash flow from operating activities?",
    "What was Tesla's revenue?",   # not an exact predefined question
]
for q in predefined:
    print(f"You: {q}\nBot: {simple_chatbot(q)}\n")

You: What is the total revenue?
Bot: Microsoft's total revenue in FY2025 was $281.7bn.

You: How has net income changed over the last year?
Bot: Microsoft's net income rose 15.5% in FY2025.

You: What are the total assets?
Bot: Microsoft's total assets were $619.0bn at the end of FY2025.

You: What are the total liabilities?
Bot: Microsoft's total liabilities were $275.5bn at the end of FY2025.

You: What is the cash flow from operating activities?
Bot: Microsoft's cash flow from operating activities in FY2025 was $136.2bn.

You: What was Tesla's revenue?
Bot: Sorry, I can only provide information on predefined queries.



**What version 1 shows.** It works, but only for exact wording, one company and the latest year. The last question, which any client might ask, fails. For GFC's diverse client base, that would not be user-friendly enough, so I built version 2.

## 4. Version 2: how the GFC Financial Chatbot works
Every message goes through a five-step pipeline:

| Step | What happens | Example: *"How has Tesla's profit changed over the last year?"* |
|---|---|---|
| 1. Normalise | Lower case, remove punctuation | `how has tesla's profit changed over the last year` |
| 2. Extract entities | Find company, year and metric using keyword dictionaries | Company = Tesla, Year = FY2025 (latest), Metric = net income ("profit") |
| 3. Classify intent | Value, change, trend, comparison, summary or help | Intent = change ("changed") |
| 4. Use memory | Fill gaps from the previous turn for follow-ups | Not needed here |
| 5. Answer | Look up the verified data, write plain English, add context | Net income fell 46.5%, plus the reasons Tesla gave |

**Design choices linked to GFC's requirements:**
- **Accuracy:** every figure comes from the verified 10-K dataset. The bot declines questions about companies or years it has no data for, rather than guessing.
- **User-friendly:** synonyms are understood ("profit", "earnings" and "bottom line" all mean net income), figures appear in billions, and ratios come with a one-line explanation.
- **Engagement:** each answer suggests two related questions, and charts can be drawn on request.
- **Context:** one-off items such as Apple's FY2024 tax charge are explained automatically, so users are not misled by a bare percentage.
- **Scalability:** the bot reads companies and years from the data file, so adding a company or a year means adding rows, not changing code.

### How the build applies the three principles of chatbot development

| Principle | What it means | Where it is in the code |
|---|---|---|
| **Rule-based logic** | If the user asks X, respond with Y | Version 1 maps exact questions with `if`/`elif`. Version 2 uses rules too, but on *meaning* rather than exact wording: keyword dictionaries identify the company, year, metric and intent, and each intent has its own response rule (`_answer_value`, `_answer_change`, `_answer_trend`, `_answer_compare`, `_answer_summary`). |
| **State management** | Remember the conversation's context | The `Memory` object stores the last company, metric, year and intent. "What about Microsoft?" or "And in 2024?" borrow whatever they leave out. A complete new question replaces the memory, so old context never leaks into an unrelated answer. |
| **Error handling** | Handle unrecognised queries gracefully and guide the user | Unrecognised questions get "Sorry, I didn't understand that" with a pointer to `help`. Companies or years outside the data are declined by name. A change for the first year explains why it can't be calculated and suggests a year that works. Empty messages are caught. |

In [6]:
print("Metrics the bot understands, and some of the words people use for them:\n")
for key, m in METRICS.items():
    print(f"  {m['label']:<30} {', '.join(m['keywords'][:5])}")

Metrics the bot understands, and some of the words people use for them:

  total revenue                  total revenue, revenue, revenues, sales, turnover
  net income                     net income, net profit, profit, profits, earnings
  total assets                   total assets, assets
  total liabilities              total liabilities, liabilities, debt, debts, obligations
  operating cash flow            cash flow from operating activities, operating cash flow, cash from operations, cash flow, operating cash
  net profit margin              net profit margin, profit margin, net margin, margin, margins
  liabilities-to-assets ratio    liabilities to assets, debt ratio, leverage, leveraged, gearing
  shareholders' equity           shareholders equity, shareholders' equity, equity, net worth, book value


## 5. Version 2 in action: a sample conversation
The same conversation a GFC client might have. Note the follow-up questions, which rely on the bot remembering the topic.

In [7]:
bot = FinancialChatbot()

conversation = [
    "Hello",
    "What was Apple's revenue in 2025?",
    "How has Tesla's net income changed over the last year?",
    "What about Microsoft?",
    "And in 2024?",
    "Which company has the highest net margin?",
    "Which company is least leveraged?",
    "Show Microsoft's operating cash flow trend",
    "How healthy is Tesla?",
    "Apple net income 2024",
]
for q in conversation:
    print(f"You: {q}\nBot: {bot.respond(q)}\n")

You: Hello
Bot: Hello! I'm the GFC Financial Chatbot. Ask me about Microsoft, Tesla or Apple, or type 'help'.

You: What was Apple's revenue in 2025?
Bot: Apple's total revenue in FY2025 was $416.2bn, which rose 6.4% on FY2024.
  You could also ask: "How has Apple's total revenue changed?" or "Which company has the highest total revenue?"

You: How has Tesla's net income changed over the last year?
Bot: Tesla's net income fell 46.5% in FY2025, from $7.1bn to $3.8bn.
  Context: Tesla linked the fall to lower vehicle deliveries and regulatory credit revenue, tariffs, and higher spending on AI and R&D.
  You could also ask: "Show Tesla's net income trend" or "What about Apple?"

You: What about Microsoft?
Bot: Microsoft's net income rose 15.5% in FY2025, from $88.1bn to $101.8bn.
  You could also ask: "Show Microsoft's net income trend" or "What about Apple?"

You: And in 2024?
Bot: Microsoft's net income rose 21.8% in FY2024, from $72.4bn to $88.1bn.
  You could also ask: "Show Microsoft

## 6. Communicating complex financial insights
The chatbot is designed for GFC clients who may have no financial background.

**Simplification and summarisation.** Figures are shown in billions rather than raw millions, ratios come with a one-line plain-English meaning, and a health summary condenses each company into a verdict and four key facts.

In [8]:
bot = FinancialChatbot()
print(bot.respond("How healthy is Microsoft?"))
print()
print(bot.respond("What is Apple's leverage?"))

Microsoft: financial health summary (FY2023 to FY2025)
  Verdict: Strong and improving: the fastest growth, highest margins and falling leverage of the three.
  Revenue: $281.7bn in FY2025 (average change +15.3% a year).
  Net income: $101.8bn, a net margin of 36.1%.
  Operating cash flow: $136.2bn (1.3x net income).
  Leverage: liabilities fund 44.5% of assets.
  You could also ask: "Show Microsoft's net income trend" or "Which company has the highest net margin?"

Apple's liabilities-to-assets ratio in FY2025 was 79.5%. This measures the share of the company's assets funded by liabilities; lower means less leveraged.
  Context: Apple's high ratio is a deliberate choice: it returns large amounts of cash to shareholders through buybacks and dividends, which keeps equity low. It is not a sign of distress.
  You could also ask: "How has Apple's liabilities-to-assets ratio changed?" or "Which company has the highest liabilities-to-assets ratio?"


**Interactive dialogue design.** Every answer ends with two suggested follow-up questions, chosen to fit what the user just asked. After a ranking, the suggestion is about the company that came first. This encourages users to explore rather than stop after one question.

In [9]:
bot = FinancialChatbot()
print(bot.respond("Which company has the highest net margin?"))
print()
print(bot.respond("How healthy is Microsoft?"))

Net profit margin in FY2025 (highest first):
  1. Microsoft: 36.1%
  2. Apple: 26.9%
  3. Tesla: 4.0%
Microsoft has the highest net profit margin. (Net profit margin measures the share of each dollar of revenue kept as profit.)
  You could also ask: "How healthy is Microsoft?" or "Show a chart of net profit margin"

Microsoft: financial health summary (FY2023 to FY2025)
  Verdict: Strong and improving: the fastest growth, highest margins and falling leverage of the three.
  Revenue: $281.7bn in FY2025 (average change +15.3% a year).
  Net income: $101.8bn, a net margin of 36.1%.
  Operating cash flow: $136.2bn (1.3x net income).
  Leverage: liabilities fund 44.5% of assets.
  You could also ask: "Show Microsoft's net income trend" or "Which company has the highest net margin?"


**Visual aids.** Asking for a chart produces a text bar chart that works in any chat window, with a sentence explaining how to read it and a pointer to the full line chart saved in Task 1. In a web version, the same request would display the chart image itself.

In [10]:
bot = FinancialChatbot()
print(bot.respond("Show a chart of revenue"))

Total revenue by fiscal year (each block is about $13.9bn):
  Apple
    FY2023 ████████████████████████████ $383.3bn
    FY2024 ████████████████████████████ $391.0bn
    FY2025 ██████████████████████████████ $416.2bn
  Microsoft
    FY2023 ███████████████ $211.9bn
    FY2024 ██████████████████ $245.1bn
    FY2025 ████████████████████ $281.7bn
  Tesla
    FY2023 ███████ $96.8bn
    FY2024 ███████ $97.7bn
    FY2025 ███████ $94.8bn
Reading the chart: longer bars mean higher values. Over three years: Apple up, Microsoft up, Tesla down.
A full line chart of this is saved in the project at charts/revenue_trend.png.
  You could also ask: "Which company has the highest total revenue?" or "How healthy is Microsoft?"


## 7. Handling questions it cannot answer
For a financial tool, saying "I don't know" is far better than inventing a figure.

In [11]:
bot = FinancialChatbot()
for q in ["What is Amazon's revenue?", "What was Microsoft's revenue in 2019?",
          "What is the weather today?", "How did Apple's revenue change in 2023?"]:
    print(f"You: {q}\nBot: {bot.respond(q)}\n")

You: What is Amazon's revenue?
Bot: I don't have data for Amazon. I currently cover Microsoft, Tesla and Apple for FY2023 to FY2025.

You: What was Microsoft's revenue in 2019?
Bot: My data covers FY2023 to FY2025, so I can't answer for 2019. I won't guess figures I haven't verified.

You: What is the weather today?
Bot: Sorry, I didn't understand that. I answer questions about the financial performance of Microsoft, Tesla and Apple. Type 'help' for examples.

You: How did Apple's revenue change in 2023?
Bot: FY2023 is the first year in my data, so I can't calculate a change for it. Try asking about FY2024 or FY2025.
  You could also ask: "Show Apple's total revenue trend" or "What about Microsoft?"



## 8. Help and guidance for new users

In [12]:
print(FinancialChatbot().respond("help"))

I can answer questions about Microsoft, Tesla and Apple, using their 10-K filings for FY2023 to FY2025.
  Metrics: revenue, net income, total assets, total liabilities, operating cash flow, net margin, leverage and shareholders' equity.
  Try asking:
  - What was Apple's revenue in 2025?
  - How has Tesla's net income changed over the last year?
  - Show Microsoft's operating cash flow trend
  - Which company has the highest net margin?
  - How healthy is Tesla?
  - Show a chart of revenue
  - What about Microsoft?  (follow-ups remember what we were discussing)
  Note: I report figures from company filings. This is not investment advice.


## 9. Testing
The test suite checks that answers match the dataset, follow-ups work, one-off items are explained and out-of-scope questions are declined.

In [13]:
import subprocess
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", "tests"], capture_output=True, text=True, cwd="..")
print(result.stdout[-300:])

................................                                         [100%]
32 passed in 0.38s



## 10. Limitations
Being clear about limits is part of delivering a reliable tool:
1. **Keyword matching, not true language understanding.** Unusual phrasing, spelling mistakes or complex multi-part questions may not be understood. The bot then says so rather than guessing.
2. **Limited data.** Three companies, three fiscal years and eight metrics. It cannot answer questions about quarters, segments, share prices or forecasts.
3. **Context notes are hand-written.** They cover the main one-off items found in the MD&A, and would need maintaining as new filings arrive.
4. **No forecasting or advice.** The bot reports historical figures only. It is not investment advice.
5. **Prototype interface.** It runs in a notebook, a terminal or a simple Flask web page (`python -m chatbot.app`). A client-facing version would need accounts, security and a production front end.

## 11. My role and hand-offs to the wider team
My responsibility in the BCG team is the **rule-based logic**: the foundation that gives the chatbot accurate, immediate answers to predefined financial queries. I built it so each specialist team can extend it without rewriting it:

| Team | What they will build | Where my rule-based core hands over |
|---|---|---|
| **NLP experts** | Understanding nuanced language | Replace or extend the keyword dictionaries (`COMPANY_ALIASES`, `METRICS`, `INTENT_KEYWORDS`) with a language model. The rest of the pipeline stays the same, because it only needs a company, year, metric and intent. |
| **Machine learning engineers** | Learning from interactions, handling complex queries | Questions the bot declines ("Sorry, I didn't understand that") are exactly the training examples they need. Logging these is the first step. `data/financial_features_ml.csv` is already ML-ready. |
| **Data integration specialists** | Real-time, comprehensive data | The bot reads companies and years from the data file, so a live feed from the SEC EDGAR API only has to produce the same columns. No chatbot code changes are needed. |
| **UX designers** | An intuitive interface for any user | `respond()` takes a message and returns plain text, so it plugs straight into a web or app front end. Suggested follow-ups and chart requests are ready to become buttons and real charts. |

## 12. Next steps for GFC
1. **Automate data ingestion** from the SEC EDGAR XBRL API, so new 10-K and 10-Q filings update the bot automatically.
2. **Add a large language model (LLM) layer** for flexible understanding of any phrasing, with answers still grounded in the verified dataset (retrieval-augmented generation), so the model cannot invent figures.
3. **Build a web interface** so GFC's clients can use it without any technical knowledge.
4. **Expand coverage** to more companies, quarterly data and segment detail.

## 13. Conclusion
Version 2 meets GFC's brief: it answers in seconds rather than hours (**efficiency**), uses only verified 10-K data and refuses to guess (**accuracy**), understands everyday phrasing, explains its answers, suggests follow-ups and draws charts on request (**user-friendly**), and grows by adding data rather than code (**scalability**).